In [1]:
import os
import shutil
import random
from pathlib import Path

In [2]:
def split_dataset(root_dir: str, train_ratio: float = 0.8, seed: int = 42):
    random.seed(seed)

    dataset_path = Path(root_dir)
    image_dir = dataset_path / "images"
    mask_dir = dataset_path / "masks"

    if not image_dir.exists() or not mask_dir.exists():
        print(f"Bỏ qua {dataset_path.name} - Thiếu thư mục images hoặc masks")
        return

    # Tạo thư mục train và valid
    train_img_dir = dataset_path / "train" / "images"
    train_mask_dir = dataset_path / "train" / "masks"
    valid_img_dir = dataset_path / "valid" / "images"
    valid_mask_dir = dataset_path / "valid" / "masks"

    for d in [train_img_dir, train_mask_dir, valid_img_dir, valid_mask_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # Lấy danh sách file
    image_files = sorted(list(image_dir.iterdir()))
    mask_files = sorted(list(mask_dir.iterdir()))

    if len(image_files) != len(mask_files):
        print(f"Cảnh báo: Số lượng ảnh và mask không khớp trong {dataset_path.name}")
        return

    # Ghép đôi image và mask theo tên file
    file_pairs = list(zip(image_files, mask_files))

    # Shuffle với seed cố định
    random.shuffle(file_pairs)

    n_total = len(file_pairs)
    n_train = int(n_total * train_ratio)
    n_valid = n_total - n_train

    print(f"📁 Dataset: {dataset_path.name:12} | Tổng: {n_total:4} | "
          f"Train: {n_train:4} | Valid: {n_valid:4}")

    # Di chuyển file
    moved_count = 0
    for i, (img_path, mask_path) in enumerate(file_pairs):
        if i < n_train:
            # Di chuyển vào train
            shutil.move(str(img_path), str(train_img_dir / img_path.name))
            shutil.move(str(mask_path), str(train_mask_dir / mask_path.name))
        else:
            # Di chuyển vào valid
            shutil.move(str(img_path), str(valid_img_dir / img_path.name))
            shutil.move(str(mask_path), str(valid_mask_dir / mask_path.name))
        
        moved_count += 1
        if moved_count % 100 == 0:
            print(f"   → Đã di chuyển {moved_count}/{n_total} file")

    print(f"Hoàn tất {dataset_path.name}: Train={n_train}, Valid={n_valid}\n")

In [ ]:
base_dir = "Datasets"          # Thư mục gốc chứa tất cả dataset

datasets = ["JSRT", "LUNA", "Montgomery", "Shenzhen"]

print("="*70)
print("BẮT ĐẦU CHIA DỮ LIỆU - DI CHUYỂN TRỰC TIẾP")
print("="*70)
print(f"Base directory: {base_dir}")
print(f"Train ratio   : 0.8 | Seed: 42\n")

for ds in datasets:
    dataset_path = os.path.join(base_dir, ds)
    if os.path.exists(dataset_path):
        split_dataset(dataset_path, train_ratio=0.8, seed=42)
    else:
        print(f"Không tìm thấy dataset: {ds}")

print("="*70)
print("HOÀN TẤT CHIA TẤT CẢ DATASET!")
print("="*70)